# Spark lab (module 06, Part A) from a local notebook — Livy + sparkmagic

This notebook is the same exercise as **sections 0–10 of [`lab_basics_spark.md`](../lab_basics_spark.md)**, for participants who work
from their laptop instead of `pyspark3` on the edge node. Each `%%spark` cell holds the code you would type at the `>>>` prompt.
`sparkmagic` sends it to a **Livy session** on the cluster, where it runs in a YARN container as **you**. Your laptop needs no Spark installation and no Kerberos.

**Before you start:** set up Jupyter exactly as in [module 09, section 6](../../../09-basics-livy/lab/lab_basics_livy.md)
(Python 3.12 environment, `pip install -r ../../../09-basics-livy/lab/notebooks/requirements.txt`, and the `sparkmagic` config template copied to `~/.sparkmagic/config.json`).
You also need the **Livy URL** from your handout.

**What differs from `pyspark3`**

| `pyspark3` on the edge node | This notebook |
|---|---|
| The driver runs on the edge node (client mode) | The driver runs in a **YARN container** (Livy uses cluster mode) |
| Paths and user come from shell `export`s (`os.environ`) | The cluster session does not see your shell: section 6 sets the paths in the session |
| Output appears in the terminal | Output appears under the cell; every cell is one Livy *statement* |
| `exit()` ends the session | `%spark delete` ends it (run it at the end) |

Part B of the lab (`spark-submit`, the Metastore table, failure signals, Spark UI and logs) stays on the edge node.
The Parquet output of section 10 goes to the same place as in the lab (`/user/<you>/spark-curated`), so `./06-basics-spark/lab/check.sh` accepts it.

## How the cells work

| Cell starts with | Runs on | Meaning |
|---|---|---|
| nothing, or `%spark ...` | your **laptop** | plain Python, or session management (`add`, `info`, `delete`) |
| `%%spark -s spark_lab` | the **cluster** | PySpark code in your session `spark_lab`, like typing it at `>>>` (the last expression is echoed and `print` works) |
| `%%spark -s spark_lab -c sql` | the **cluster** | the cell is a Spark SQL query; the result comes back as a table |
| `%%spark -s spark_lab -o hourly -m take -n 100` | the cluster, then laptop | copies the DataFrame `hourly` to a **pandas** DataFrame with the same name here (at most 100 rows) |

All `%%spark` cells with the same `-s` name share one session: `spark`, variables, DataFrames and temp views stay alive from cell to cell, just like in the shell.
Run the cells **in order**. If you restart the kernel or delete the session, start again from section 1 (the variables of the session are gone).
Full explanation of all flags: top of [`livy_sparkmagic.ipynb`](../../../09-basics-livy/lab/notebooks/livy_sparkmagic.ipynb) in module 09.

## Connection and session

The password is asked with `getpass`: it stays in the memory of this kernel and is never written into the notebook file.

In [ ]:
import getpass

livy_url = input("Livy URL (from your handout, no trailing slash): ").strip().rstrip("/")
ws_user = input("User name (WORKSHOP_USER): ").strip()
ldap_password = getpass.getpass(f"LDAP password for {ws_user}: ")

In [ ]:
%load_ext sparkmagic.magics

`sparkmagic` sends your settings from `~/.sparkmagic/config.json` to Livy when it creates the session. The next cell adds a **session name** to them, before the session is created.


In [ ]:
from datetime import datetime
from sparkmagic.utils import configuration as conf

# Livy session name (shows up in Livy and Cloudera Manager). It must be unique on the Livy server,
# so it has your user name and the start time. The label after -s in the cells below is a different thing: it only exists in this notebook.
session_cfg = conf.session_configs()          # settings from ~/.sparkmagic/config.json
session_cfg["name"] = f"{ws_user}-spark-lab-{datetime.now():%H%M%S}"
conf.override(conf.session_configs.__name__, session_cfg)
session_cfg["name"]


`%spark add` creates the Livy session (the `POST /sessions` of module 09). It takes about a minute: the progress bar shows `starting`, then `idle`. `-k` makes the cell safe to run twice.

In [ ]:
%spark add -s spark_lab -l python -u $livy_url -a $ws_user -p $ldap_password -t Basic_Access -k

## 0. Orient — your session instead of `pyspark3`

**Goal:** Find out which Spark application serves your cell.

The session already provides `spark` (the `SparkSession`) and `sc`, like the shell did.

In [ ]:
%%spark -s spark_lab

spark.sparkContext.appName
spark.sparkContext.master
spark.version
spark.sparkContext.applicationId

`applicationId` is the YARN application of your session. In a terminal on the edge node (valid ticket) you find it in the list:

```bash
yarn application -list -appStates RUNNING -appTypes SPARK
```

**Check question:** With `pyspark3` the driver ran on the edge node. Where does it run for this session? Compare `%spark info` (it shows the application id and the Spark UI link) with the YARN list.

In [ ]:
%spark info

## 1. Warm-up — plain Python variables

**Goal:** Get comfortable with the cells before touching Spark. This is ordinary Python; Spark is not involved yet — but the code already runs in the **driver** on the cluster.

In [ ]:
%%spark -s spark_lab

station = "TS-101"
station

In [ ]:
%%spark -s spark_lab

power_kw = 42.5
load_pct = 63
station, power_kw, load_pct

In [ ]:
%%spark -s spark_lab

power_w = power_kw * 1000
print(f"{station} delivers {power_w} W")

A **list** holds several values, a **tuple** is a fixed group, and a **dictionary** maps names to values:

In [ ]:
%%spark -s spark_lab

powers = [42.5, 39.0, 51.2]
sum(powers) / len(powers)

In [ ]:
%%spark -s spark_lab

reading = ("TS-101", 42.5)
reading[0]

In [ ]:
%%spark -s spark_lab

reading_dict = {"station_id": "TS-101", "active_power_kw": 42.5}
reading_dict["active_power_kw"]

A list of tuples already looks like a small table — this is what we turn into a Spark DataFrame next:

In [ ]:
%%spark -s spark_lab

rows = [
    ("TS-101", 42.5, 63),
    ("TS-102", 39.0, 58),
    ("TS-103", 51.2, 71),
]
rows

**Check question:** Where does the variable `rows` live — on HDFS, in an executor, or in the driver's memory? Could you use it in a plain cell of this notebook (laptop)?

## 2. Your first DataFrame — built by hand

**Goal:** Turn the Python list into a Spark DataFrame, without reading any file.

A **DataFrame** is a table with named, typed columns. Spark spreads its rows over **partitions** so executors can process them in parallel.

In [ ]:
%%spark -s spark_lab

df = spark.createDataFrame(rows, ["station_id", "active_power_kw", "load_pct"])
df

The cell only shows the column names and types. To see data, call an **action**:

In [ ]:
%%spark -s spark_lab

df.show()

Look at the structure Spark inferred from the Python values:

In [ ]:
%%spark -s spark_lab

df.printSchema()
df.columns
df.dtypes

Count the rows and fetch them back into Python. `collect()` brings **all** rows into the driver — fine for 3 rows, dangerous for 3 billion; prefer `show()`, `take(2)` or `limit(10)` on real data.

In [ ]:
%%spark -s spark_lab

df.count()
df.collect()

Instead of letting Spark guess, state the schema yourself (a string of `name TYPE` pairs):

In [ ]:
%%spark -s spark_lab

df2 = spark.createDataFrame(
    [("TS-101", 42.5, 63), ("TS-102", 39.0, 58)],
    "station_id STRING, active_power_kw DOUBLE, load_pct INT")
df2.printSchema()
df2.show()

**Check question:** Which of `show`, `printSchema`, `count`, `collect` are actions that make Spark do work, and which only inspect metadata?

## 3. Select, filter, add columns — still tiny data

**Goal:** Learn the basic DataFrame verbs on the 3-row table. Import the column helper functions once (the alias `F` is a common convention; later sections use it too):

In [ ]:
%%spark -s spark_lab

from pyspark.sql import functions as F

**Select** columns and **filter** rows:

In [ ]:
%%spark -s spark_lab

df.select("station_id", "load_pct").show()
df.filter(F.col("load_pct") > 60).show()

**Add** a column with `withColumn`:

In [ ]:
%%spark -s spark_lab

df_kw = df.withColumn("active_power_w", F.col("active_power_kw") * 1000)
df_kw.show()

Use a condition to create a label, then sort:

In [ ]:
%%spark -s spark_lab

df_flag = df.withColumn(
    "status", F.when(F.col("load_pct") >= 70, "HIGH").otherwise("NORMAL"))
df_flag.show()
df_flag.orderBy(F.col("active_power_kw").desc()).show()

Every call returns a **new** DataFrame; `df` itself never changes:

In [ ]:
%%spark -s spark_lab

df.show()

**Lazy evaluation:** `df_kw = ...` ran no job. Only `show()`, `count()`, `collect()` or `write` trigger work. Spark records the recipe first and optimizes it. Peek at the recipe:

In [ ]:
%%spark -s spark_lab

df_flag.explain()

**Check question:** If you chain ten `filter` calls but never call `show` or `count`, does YARN start executors for them?

## 4. Aggregate — groupBy and SQL on a tiny table

**Goal:** Summarize values per group, first with the DataFrame API and then with SQL. Build a slightly larger hand-made table:

In [ ]:
%%spark -s spark_lab

tiny = spark.createDataFrame(
    [("TS-101", "08:00", 42.5), ("TS-101", "08:15", 44.0),
     ("TS-102", "08:00", 39.0), ("TS-102", "08:15", 41.5),
     ("TS-103", "08:00", 51.2), ("TS-103", "08:15", 49.8)],
    ["station_id", "time", "active_power_kw"])
tiny.show()

Average, maximum and number of readings per station:

In [ ]:
%%spark -s spark_lab

(tiny.groupBy("station_id")
     .agg(F.avg("active_power_kw").alias("avg_kw"),
          F.max("active_power_kw").alias("max_kw"),
          F.count(F.lit(1)).alias("n"))
     .orderBy("station_id")
     .show())

Register the DataFrame as a **temp view** and ask the same question in SQL — same engine, same plan:

In [ ]:
%%spark -s spark_lab

tiny.createOrReplaceTempView("tiny_readings")
spark.sql("""
SELECT station_id,
       AVG(active_power_kw) AS avg_kw,
       MAX(active_power_kw) AS max_kw,
       COUNT(*)             AS n
FROM tiny_readings
GROUP BY station_id
ORDER BY station_id
""").show()

You should see three rows (`TS-101`, `TS-102`, `TS-103`) with `n = 2`.

**Check question:** Why is a temp view gone when the session ends, and does it write anything to the Metastore?

## 5. Join — two tiny tables

**Goal:** Combine two DataFrames on a common column. A second hand-made table with station names:

In [ ]:
%%spark -s spark_lab

names = spark.createDataFrame(
    [("TS-101", "Nord"), ("TS-102", "Süd"), ("TS-104", "West")],
    ["station_id", "region"])
names.show()

An **inner** join keeps only stations present in both tables; a **left** join keeps all stations from the left table and fills missing values with `null`. Predict the output before you run it: which station is missing from the inner join? Which `region` is `null` in the left join?

In [ ]:
%%spark -s spark_lab

df.join(names, "station_id", "inner").show()
df.join(names, "station_id", "left").show()

**Check question:** What changes if you swap the two tables in the left join?

## 6. Read CSV — schema on read

**Goal:** Now use real files: load smart-meter and operator weather CSVs from HDFS and inspect types.

In `pyspark3` the paths came from `os.environ`. The session on the cluster does not see your shell, so set them here. `WORKSHOP_ROOT` is the workshop folder of your handout (default below); the user and the home folder follow from the user the session runs as.

In [ ]:
%%spark -s spark_lab

import getpass

workshop_root = "/2026_cdp_workshop"          # change only if your handout says otherwise
workshop_user = getpass.getuser()             # the session runs as you
workshop_home = "/user/" + workshop_user
meter_csv   = workshop_root + "/data/grid_measurements/sample-grid-measurements.csv"
weather_csv = workshop_root + "/data/operator_weather/sample-operator-weather.csv"
awattar_csv = workshop_root + "/data/awattar/sample-awattar-prices.csv"
print(workshop_user, workshop_home)

Read the meter file with a header and inferred types (small data — inference is fine here):

In [ ]:
%%spark -s spark_lab

meter_raw = spark.read.option("header", True).option("inferSchema", True).csv(meter_csv)
meter_raw.printSchema()
meter_raw.show(5, False)
meter_raw.count()

The first column is the reading time; the CSV header calls it `timestamp`. For production pipelines you prefer an explicit schema (as in section 2), so a bad row cannot widen a column type. With a schema, Spark maps CSV fields **by position** — so name the first field `ts_utc` even though the header says `timestamp`:

In [ ]:
%%spark -s spark_lab

meter = (spark.read.option("header", True)
    .schema(
        "ts_utc STRING, station_id STRING, device_id STRING, phase STRING, "
        "active_power_kw DOUBLE, reactive_power_kvar DOUBLE, voltage_v DOUBLE, "
        "current_a DOUBLE, power_factor DOUBLE, frequency_hz DOUBLE, "
        "transformer_load_pct DOUBLE, quality_flag STRING, event_code STRING, "
        "comm_status STRING")
    .csv(meter_csv)
    .withColumn("ts_utc", F.to_timestamp(F.col("ts_utc"))))
meter.count()

Load weather the same way (two operator stations, 15-minute steps):

In [ ]:
%%spark -s spark_lab

weather = (spark.read.option("header", True)
    .schema(
        "ts_utc STRING, weather_station_id STRING, station_name STRING, "
        "temperature_c DOUBLE, relative_humidity_pct INT, wind_speed_mps DOUBLE, "
        "precipitation_mm DOUBLE, pressure_hpa DOUBLE, weather_quality STRING")
    .csv(weather_csv)
    .withColumn("ts_utc", F.to_timestamp(F.col("ts_utc"))))
weather.count()

You should see **15** meter rows and **10** weather rows (header excluded).

**Check question:** Why is an explicit schema safer than `inferSchema=True` on a large landing folder?

## 7. Cleanse — filters and lazy evaluation

**Goal:** Keep quality-approved readings, normalize empty event codes, and see how actions trigger work. The assignment returns instantly without launching a job:

In [ ]:
%%spark -s spark_lab

meter_ok = (meter.filter(F.col("quality_flag") == "OK")
    .withColumn("event_code",
        F.when(F.trim(F.col("event_code")) == "", F.lit(None)).otherwise(F.col("event_code")))
    .dropDuplicates(["ts_utc", "station_id", "device_id", "phase"]))

In [ ]:
%%spark -s spark_lab

rejected = meter.count() - meter_ok.count()
print(f"rejected_rows={rejected}")
meter_ok.count()

You should see **3** rejected rows (WARN quality flags in the sample) and **12** cleansed rows. The first `count()` on `meter_ok` schedules a new job — you can follow it in the Spark UI (link in `%spark info`).

## 8. Aggregate — DataFrame API and Spark SQL

**Goal:** Summarize average power and peak load per transformer station and hour (same pattern as section 4).

In [ ]:
%%spark -s spark_lab

hourly = (meter_ok
    .withColumn("hour_utc", F.date_trunc("hour", F.col("ts_utc")))
    .groupBy("station_id", "hour_utc")
    .agg(F.avg("active_power_kw").alias("avg_power_kw"),
         F.max("transformer_load_pct").alias("max_load_pct"),
         F.count(F.lit(1)).alias("reading_count"))
    .orderBy("station_id", "hour_utc"))
hourly.show(truncate=False)

Register a temp view and express the same logic in SQL:

In [ ]:
%%spark -s spark_lab

meter_ok.createOrReplaceTempView("meter_ok")
spark.sql("""
SELECT station_id,
       date_trunc('hour', ts_utc) AS hour_utc,
       AVG(active_power_kw) AS avg_power_kw,
       MAX(transformer_load_pct) AS max_load_pct,
       COUNT(*) AS reading_count
FROM meter_ok
GROUP BY station_id, date_trunc('hour', ts_utc)
ORDER BY station_id, hour_utc
""").show(truncate=False)

You should see two hour buckets (08:00 and 09:00 UTC) for each of the three transformer stations.

With sparkmagic the same query can be a **SQL cell** (`-c sql`); the result is rendered as a table:

In [ ]:
%%spark -s spark_lab -c sql

SELECT station_id, date_trunc('hour', ts_utc) AS hour_utc, ROUND(AVG(active_power_kw), 2) AS avg_power_kw
FROM meter_ok
GROUP BY station_id, date_trunc('hour', ts_utc)
ORDER BY station_id, hour_utc

**Bonus, only possible in a notebook:** copy the small result `hourly` to your laptop as a **pandas** DataFrame (`-o hourly`, at most 100 rows). The cell body only has to mention the variable.
Do this only for small results — the rows travel through Knox and Livy as JSON.

In [ ]:
%%spark -s spark_lab -o hourly -m take -n 100
hourly

In [ ]:
print(type(hourly))
hourly

## 9. Join — align timestamps

**Goal:** Attach operator weather to each meter reading at the same 15-minute timestamp, and add hourly market prices.

The sample weather stations (`GRID-WS-01`, `GRID-WS-02`) differ from the transformer IDs (`TS-101`, …) — you join on **time** only, so each meter row matches **both** weather stations at that timestamp:

In [ ]:
%%spark -s spark_lab

joined = meter_ok.join(weather, ["ts_utc"], "inner")
joined.count()

You should get **24** rows (12 meter × 2 weather stations per timestamp). Try a **left** join and compare:

In [ ]:
%%spark -s spark_lab

meter_ok.join(weather, ["ts_utc"], "left").count()

Add hourly **awattar** prices by truncating the meter time to the hour:

In [ ]:
%%spark -s spark_lab

prices = (spark.read.option("header", True)
    .schema(
        "price_start STRING, price_end STRING, market STRING, "
        "price_eur_mwh DOUBLE, price_ct_kwh DOUBLE, currency STRING")
    .csv(awattar_csv)
    .withColumn("price_hour", F.date_trunc("hour", F.to_timestamp(F.col("price_start")))))
curated = (joined
    .withColumn("price_hour", F.date_trunc("hour", F.col("ts_utc")))
    .join(prices.select("price_hour", "price_eur_mwh", "market"), ["price_hour"], "left")
    .drop("price_hour"))
curated.show(5, False)

**Check question:** With an inner join on time only, why does one meter timestamp produce two output rows?

## 10. Write Parquet — curated layer on HDFS

**Goal:** Persist the joined dataset as partitioned Parquet under your HDFS home.

In [ ]:
%%spark -s spark_lab

out_interactive = workshop_home + "/spark-curated"
curated.write.mode("overwrite").partitionBy("station_id").parquet(out_interactive)
print("written to", out_interactive)

Read it back to check (24 rows expected):

In [ ]:
%%spark -s spark_lab

spark.read.parquet(out_interactive).count()

From a terminal on the edge node inspect the layout, as in the lab:

```bash
hdfs dfs -ls "/user/${WORKSHOP_USER}/spark-curated"
hdfs dfs -ls "/user/${WORKSHOP_USER}/spark-curated/station_id=TS-101"
hdfs dfs -du -s -h "/user/${WORKSHOP_USER}/spark-curated"
```

You should see `_SUCCESS`, one directory per `station_id`, and Parquet part files inside. Only these files survive the session; `meter`, `curated`, … live in the Spark driver and are gone when the session ends.

## Close the session

Closing the notebook does **not** stop the Livy session — the Spark application keeps running on YARN until Livy's idle timeout, and blocks containers for others. `%spark delete` is the `DELETE /sessions/<id>` of module 09. Run it every time.

In [ ]:
%spark delete -s spark_lab

Continue with **Part B** of [`lab_basics_spark.md`](../lab_basics_spark.md) on the edge node (`spark-submit`, Metastore table, failure signals, Spark UI and logs).

**Question:** Which of your variables survived `%spark delete`, and which files did? How is that different from leaving `pyspark3` with `exit()`?